In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, adjusted_rand_score)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
plt.rcParams["figure.figsize"] = (8, 4)
print("TensorFlow:", tf.__version__)
print("GPU disponível:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
from tensorflow import keras

pasta = "/content/dataset"

treino, validacao = keras.utils.image_dataset_from_directory(
    pasta,
    image_size=(96, 96),
    batch_size=32,
    validation_split=0.2,
    subset="both",
    seed=42
)

print("Classes:", treino.class_names)

In [ ]:
x_treino =
# Separando treino e validação (80/20) — o TESTE oficial do CIFAR-10 fica guardado, intocado, até a Parte 10.
x_treino = x_treino, x_val, y_treino, y_val = train_test_split(
    x_full, y_full,
    test_size=0.20,
    shuffle=True,            # embaralha antes de dividir
    stratify=y_full,         # preserva a proporção das classes
    random_state=SEED)


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)
base.trainable = False

modelo = keras.Sequential([
    keras.Input(shape=(96, 96, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.Rescaling(1./127.5, offset=-1),
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(2, activation="softmax")
])

modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

In [ ]:
import time
from tensorflow import keras

parar_cedo = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

inicio = time.time()

hist = modelo.fit(
    treino,
    validation_data=validacao,
    epochs=30,
    callbacks=[parar_cedo],
    verbose=1
)

print(f"Tempo de treinamento: {(time.time() - inicio)/60:.1f} minutos")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import shutil
import time

inicio = time.time()

shutil.copytree(
    pasta,
    "/content/dataset",
    dirs_exist_ok=True
)

print(f"Cópia concluída em {(time.time() - inicio)/60:.1f} minutos!")

In [ ]:
# Avaliar o modelo treinado
perda, acuracia = modelo.evaluate(validacao, verbose=1)

print(f"Acurácia na validação: {acuracia:.2%}")

# Salvar o modelo no Google Drive
modelo.save("/content/drive/MyDrive/face_car/cnn_sonolencia.keras")

print("Modelo salvo com sucesso!")

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/face_car/cnn_sonolencia.keras"
)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Caminho do modelo V1
caminho_modelo = "/content/drive/MyDrive/face_car/cnn_sonolencia.keras"

# Carregar sem modificar o modelo
modelo = keras.models.load_model(
    caminho_modelo,
    compile=False
)

# Coletar imagens e rótulos juntos,
# garantindo a correspondência correta
y_real = []
y_previsto = []

for imagens, rotulos in validacao:
    previsoes = modelo.predict(imagens, verbose=0)
    y_previsto.extend(np.argmax(previsoes, axis=1))
    y_real.extend(rotulos.numpy())

y_real = np.array(y_real)
y_previsto = np.array(y_previsto)

nomes = ["Drowsy", "Non Drowsy"]

print("ACURÁCIA:", round(
    accuracy_score(y_real, y_previsto) * 100, 2
), "%")

print("\nRELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(
    y_real,
    y_previsto,
    target_names=nomes,
    zero_division=0
))

matriz = confusion_matrix(y_real, y_previsto)

ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=nomes
).plot(cmap="Blues")

plt.title("Matriz de Confusão - Face-Car V1")
plt.savefig(
    "/content/drive/MyDrive/face_car/matriz_confusao_v1.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


